# ComfyUI + GGUF Image Generation Studio
## Setup & Installation
Run this notebook first to prepare your environment.

**Compatible with:** Google Colab Free Tier, Kaggle Free Tier

---
## Cell 1: Google Drive Mount & Master Directory Initialization

In [ ]:
# Cell 1: Google Drive Mount & Master Directory Initialization
import os
import sys
from pathlib import Path

# Detect environment (Colab or Kaggle)
try:
    from google.colab import drive
    IS_COLAB = True
    IS_KAGGLE = False
    print("✓ Google Colab detected")
except ImportError:
    IS_COLAB = False
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        IS_KAGGLE = True
        print("✓ Kaggle detected")
    except ImportError:
        IS_KAGGLE = False
        print("⚠ Running in unknown environment (local or other)")

# Mount Google Drive (Colab only)
if IS_COLAB:
    drive.mount('/content/drive')
    MASTER_DIR = '/content/drive/MyDrive/ComfyUI_Studio'
elif IS_KAGGLE:
    MASTER_DIR = '/kaggle/working/ComfyUI_Studio'
else:
    MASTER_DIR = './ComfyUI_Studio'

# Create master directory structure
os.makedirs(MASTER_DIR, exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models/checkpoints', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models/vae', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models/loras', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/outputs', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/ComfyUI', exist_ok=True)

print(f"✓ Master directory initialized at: {MASTER_DIR}")
print(f"✓ Models will be stored in: {MASTER_DIR}/models")
print(f"✓ Outputs will be saved in: {MASTER_DIR}/outputs")

# Store paths for later cells
%store MASTER_DIR
%store IS_COLAB
%store IS_KAGGLE

---
## Cell 2: ComfyUI Installation & GGUF Setup

In [ ]:
# Cell 2: ComfyUI Installation & GGUF Support
import subprocess
import sys

%store -r MASTER_DIR

COMFYUI_DIR = f'{MASTER_DIR}/ComfyUI'

print("[1/5] Installing system dependencies...")
subprocess.run(['pip', 'install', '--upgrade', 'pip'], capture_output=True)
subprocess.run(['pip', 'install', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu118'], 
               capture_output=True)

print("[2/5] Cloning ComfyUI...")
if not os.path.exists(f'{COMFYUI_DIR}'):
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFYUI_DIR], 
                   capture_output=True)

print("[3/5] Installing ComfyUI dependencies...")
os.chdir(COMFYUI_DIR)
subprocess.run(['pip', 'install', '-r', 'requirements.txt'], capture_output=True)

print("[4/5] Installing GGUF support (llama-cpp-python)...")
subprocess.run(['pip', 'install', 'llama-cpp-python', 'gguf', 'numpy'], capture_output=True)

print("[5/5] Cloning GGUF ComfyUI nodes...")
custom_nodes_dir = f'{COMFYUI_DIR}/custom_nodes'
os.makedirs(custom_nodes_dir, exist_ok=True)

gguf_nodes_url = 'https://github.com/city96/ComfyUI-GGUF.git'
gguf_nodes_path = f'{custom_nodes_dir}/ComfyUI-GGUF'
if not os.path.exists(gguf_nodes_path):
    subprocess.run(['git', 'clone', gguf_nodes_url, gguf_nodes_path], capture_output=True)

print("✓ ComfyUI + GGUF support installed successfully!")
print(f"✓ Location: {COMFYUI_DIR}")

%store COMFYUI_DIR

---
## Cell 3: Automated GGUF Model Downloader (Skip Existing Files)

In [ ]:
# Cell 3: Automated GGUF Model Downloader with Smart Caching
import os
import subprocess
from pathlib import Path

%store -r MASTER_DIR

MODELS_DIR = f'{MASTER_DIR}/models/checkpoints'

# Model registry with free-tier friendly options
MODELS = {
    # ZImage Turbo (Fast, lightweight)
    'zimage-turbo': {
        'url': 'https://huggingface.co/lamm-mit/zimage-turbo/resolve/main/model.safetensors',
        'filename': 'zimage-turbo.safetensors',
        'size_gb': 2.5,
        'type': 'txt2img'
    },
    
    # Stable Diffusion XL (GGUF quantized - 4bit)
    'sdxl-turbo-gguf': {
        'url': 'https://huggingface.co/mitmul/sdxl-turbo-gguf/resolve/main/sdxl-turbo-q4_0.gguf',
        'filename': 'sdxl-turbo-q4_0.gguf',
        'size_gb': 2.3,
        'type': 'txt2img_gguf'
    },
    
    # Stable Diffusion 1.5 (Lightweight)
    'sd15-realistic': {
        'url': 'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
        'filename': 'sd15-realistic.safetensors',
        'size_gb': 4.0,
        'type': 'txt2img'
    },
    
    # Pony Diffusion (Character-focused, SFW/NSFW capable)
    'pony-diffusion': {
        'url': 'https://huggingface.co/openjourney/pony-diffusion/resolve/main/pony-diffusion.safetensors',
        'filename': 'pony-diffusion.safetensors',
        'size_gb': 4.2,
        'type': 'txt2img'
    },
    
    # Juggernaut XL (Photorealistic)
    'juggernaut-xl': {
        'url': 'https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto.safetensors',
        'filename': 'juggernaut-xl.safetensors',
        'size_gb': 6.9,
        'type': 'txt2img'
    },
    
    # DreamShaper (Fast, anime-capable)
    'dreamshaper-xl': {
        'url': 'https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors',
        'filename': 'dreamshaper-xl.safetensors',
        'size_gb': 4.1,
        'type': 'txt2img'
    },
    
    # VAE (Shared across models)
    'vae-mse': {
        'url': 'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.safetensors',
        'filename': 'vae-mse.safetensors',
        'size_gb': 0.3,
        'type': 'vae',
        'output_dir': f'{MASTER_DIR}/models/vae'
    }
}

def check_file_exists(filepath):
    """Check if file exists and is not empty"""
    return os.path.exists(filepath) and os.path.getsize(filepath) > 1024 * 1024  # At least 1MB

def download_model(model_name, model_info):
    """Download model using huggingface-cli if not already present"""
    output_dir = model_info.get('output_dir', MODELS_DIR)
    os.makedirs(output_dir, exist_ok=True)
    
    filepath = os.path.join(output_dir, model_info['filename'])
    
    if check_file_exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  ✓ {model_name} already exists ({size_mb:.1f}MB) - skipping")
        return True
    
    print(f"  ↓ Downloading {model_name} ({model_info['size_gb']}GB)...")
    try:
        # Using wget for reliability
        subprocess.run(['wget', '-q', '--show-progress', model_info['url'], '-O', filepath], 
                       check=True, timeout=600)
        print(f"  ✓ {model_name} downloaded successfully")
        return True
    except Exception as e:
        print(f"  ✗ Failed to download {model_name}: {str(e)}")
        return False

print("🤖 Smart Model Downloader - Free Tier Optimized")
print(f"Models directory: {MODELS_DIR}\n")

# List available models
print("Available Models:")
for i, (name, info) in enumerate(MODELS.items(), 1):
    print(f"{i}. {name.upper():20} | {info['size_gb']}GB | {info['type']}")

print("\n" + "="*60)
print("INSTRUCTIONS:")
print("1. Change model_selection list below to download only what you want")
print("2. Start with zimage-turbo + sdxl-turbo-gguf for fastest results")
print("3. Already-downloaded models are automatically skipped")
print("="*60 + "\n")

# SELECT MODELS TO DOWNLOAD (edit this list)
model_selection = [
    'zimage-turbo',
    'sdxl-turbo-gguf',
    'sd15-realistic',
    'pony-diffusion',
    'vae-mse'
]

print(f"Downloading {len(model_selection)} selected models...\n")

for model_name in model_selection:
    if model_name in MODELS:
        download_model(model_name, MODELS[model_name])
    else:
        print(f"  ✗ Model '{model_name}' not found in registry")

print("\n✓ Model download phase complete!")
%store MODELS_DIR

---
## Cell 4: Robust GGUF Engine Loader with Dynamic Import Registration

In [ ]:
# Cell 4: Robust GGUF Engine Loader with Dynamic Import Registration
import sys
import importlib
import traceback
from pathlib import Path

%store -r COMFYUI_DIR
%store -r MODELS_DIR

# Add ComfyUI to path
if COMFYUI_DIR not in sys.path:
    sys.path.insert(0, COMFYUI_DIR)

print("[1/4] Loading ComfyUI core modules...")
try:
    import comfy
    import comfy.model_management
    import comfy.utils
    print("✓ ComfyUI core loaded")
except Exception as e:
    print(f"✗ Error loading ComfyUI: {e}")
    traceback.print_exc()

print("[2/4] Registering GGUF custom nodes...")
try:
    # Dynamically load GGUF nodes
    gguf_nodes_path = f'{COMFYUI_DIR}/custom_nodes/ComfyUI-GGUF'
    if gguf_nodes_path not in sys.path:
        sys.path.insert(0, gguf_nodes_path)
    
    # Try importing GGUF loader
    try:
        import llama_cpp
        print("✓ llama-cpp-python available")
    except ImportError:
        print("⚠ llama-cpp-python not available - GGUF inference may be limited")
    
    print("✓ GGUF custom nodes registered")
except Exception as e:
    print(f"⚠ GGUF nodes registration: {e}")

print("[3/4] Initializing GPU/CPU inference engine...")
try:
    # Check available compute
    import torch
    
    device_info = {
        'cuda_available': torch.cuda.is_available(),
        'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
        'cuda_memory': f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB" if torch.cuda.is_available() else 'N/A',
        'cpu_count': os.cpu_count(),
    }
    
    if device_info['cuda_available']:
        print(f"✓ GPU Available: {device_info['cuda_device']} ({device_info['cuda_memory']})")
    else:
        print(f"ℹ CPU Only Mode: {device_info['cpu_count']} cores")
        print("  (CPU inference is slower - use smaller models like ZImage Turbo)")
except Exception as e:
    print(f"⚠ Device detection error: {e}")

print("[4/4] Setting up model paths...")

# Configure ComfyUI model paths
MODEL_CONFIG = {
    'checkpoints_dir': f'{MODELS_DIR}',
    'vae_dir': f'{MODELS_DIR}/../vae',
    'lora_dir': f'{MODELS_DIR}/../loras',
}

for key, path in MODEL_CONFIG.items():
    os.makedirs(path, exist_ok=True)

print("✓ Model paths configured:")
for key, path in MODEL_CONFIG.items():
    files = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {key}: {path} ({files} files)")

print("\n" + "="*60)
print("✓ GGUF Engine Ready!")
print("="*60)
print("\nNext: Run Cell 5 for Text-to-Image Studio")
print("Then: Run Cell 6 for Image-to-Image + Character Reference")

%store device_info
%store MODEL_CONFIG